In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [5]:
torch.cuda.is_available()

True

In [6]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct", cache_dir="/hpcstor6/scratch01/h/huuthanhvy.nguyen001")

In [8]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", 
                                             device_map="cuda",
                                             cache_dir="/hpcstor6/scratch01/h/huuthanhvy.nguyen001")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [9]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [24]:
import json

with open ("/home/huuthanhvy.nguyen001/newmodel/a1_gen_chatgpt_p0.json", "r") as file:
    data1 = json.load(file)

with open ("/home/huuthanhvy.nguyen001/newmodel/a1_tune_chatgpt_p0.json", "r") as file:
    data2 = json.load(file)

In [25]:
generate_essay = data1["result"]
tune_essay = data2["result"]

In [26]:
rubric = {
    "issues": {
        "4": "Issue is stated clearly and described comprehensively, delivering all relevant information necessary for full understanding.",
        "3": "Issue is stated, described, and clarified so that understanding is not seriously impeded by omissions.",
        "2": "Issue is stated but description leaves some terms undefined, ambiguities unexplored, boundaries undetermined, and/or backgrounds unknown.",
        "1": "Issue is stated without clarification or description."
    },
    "evidence": {
        "4": "Information is taken from source(s) with enough interpretation/evaluation to develop a comprehensive analysis or synthesis. Viewpoints of experts are questioned thoroughly.",
        "3": "Information is taken from source(s) with enough interpretation/evaluation to develop a coherent analysis or synthesis. Viewpoints of experts are subject to questioning.",
        "2": "Information is taken from source(s) with some interpretation/evaluation, but not enough to develop a coherent analysis or synthesis. Viewpoints of experts are taken as mostly fact, with little questioning.",
        "1": "Information is taken from source(s) without any interpretation/evaluation. Viewpoints of experts are taken as fact, without question."
    },
    "assumptions": {
        "4": "Thoroughly (systematically and methodically) analyzes own and others' assumptions and carefully evaluates the relevance of contexts when presenting a position.",
        "3": "Identifies own and others' assumptions and several relevant contexts when presenting a position. Questions some assumptions.",
        "2": "Identifies several relevant contexts when presenting a position. May be more aware of others' assumptions than one's own. Shows emerging awareness of present assumptions.",
        "1": "Shows an emerging awareness of present assumptions (sometimes labels assertions as assumptions). Begins to identify some contexts when presenting a position."
    },
    "position": {
        "4": "Specific position is imaginative, taking into account the complexities of an issue. Limits of position are acknowledged. Others' points of view are synthesized within position.",
        "3": "Specific position takes into account the complexities of an issue. Others' points of view are acknowledged within position.",
        "2": "Specific position acknowledges different sides of an issue.",
        "1": "Specific position is stated, but is simplistic and obvious."
    },
    "conclusions": {
        "4": "Conclusions and related outcomes are logical and reflect student's informed evaluation and ability to place evidence and perspectives discussed in priority order.",
        "3": "Conclusion is logically tied to a range of information, including opposing viewpoints; related outcomes are identified clearly.",
        "2": "Conclusion is logically tied to information (chosen to fit the desired conclusion); some related outcomes are identified clearly.",
        "1": "Conclusion is inconsistently tied to some of the information discussed; related outcomes are oversimplified."
    }
}


In [29]:
messages1 = [
    {
        "role": "system",
        "content": f""" Score the essay on these 5 dimensions, each from 1 to 4:

Rubric:
{json.dumps(rubric, indent=2)}

Scoring scale:
1 = Benchmark
2 = Milestone (lower)
3 = Milestone (upper)
4 = Capstone

Return ONLY a JSON object, no explanation:
{{"issues": <1-4>, "evidence": <1-4>, "assumptions": <1-4>, "position": <1-4>, "conclusions": <1-4>}}"""
    },
    {
        "role": "user",
        "content": f"""Score this essay:\n\n{generate_essay}"""
    },
]

In [30]:
messages2 = [
    {
        "role": "system",
        "content": f""" Score the essay on these 5 dimensions, each from 1 to 4:

Rubric:
{json.dumps(rubric, indent=2)}

Scoring scale:
1 = Benchmark
2 = Milestone (lower)
3 = Milestone (upper)
4 = Capstone

Return ONLY a JSON object, no explanation:
{{"issues": <1-4>, "evidence": <1-4>, "assumptions": <1-4>, "position": <1-4>, "conclusions": <1-4>}}"""
    },
    {
        "role": "user",
        "content": f"""Score this essay:\n\n{tune_essay}"""
    },
]

In [31]:
inputs1 = tokenizer.apply_chat_template(
	messages1,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

In [34]:
inputs2 = tokenizer.apply_chat_template(
	messages2,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

In [35]:
outputs1 = model.generate(**inputs1, max_new_tokens=40)
outputs2 = model.generate(**inputs2, max_new_tokens=40)

In [36]:
print(tokenizer.decode(outputs1[0][inputs1["input_ids"].shape[-1]:]))  

{"issues": 4, "evidence": 3, "assumptions": 3, "position": 3, "conclusions": 3}<|im_end|>


In [38]:
print(tokenizer.decode(outputs2[0][inputs2["input_ids"].shape[-1]:]))  

{"issues": 4, "evidence": 4, "assumptions": 3, "position": 3, "conclusions": 4}<|im_end|>
